In [ ]:

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from pandas.api.types import CategoricalDtype

from category_encoders import MEstimateEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import KFold, cross_val_score
from xgboost import XGBRegressor



plt.style.use("seaborn-v0_8-whitegrid")

plt.rc("figure", autolayout=True)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)

# Mute warnings
warnings.filterwarnings('ignore')


In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

X = df_train.copy()
y = X.pop("SalePrice")
test_ids = df_test["Id"].copy()
test_ids.to_csv("test_ids.csv")

In [3]:
test_ids = df_test["Id"].copy()

In [ ]:
test_ids.to_csv("test_ids.csv")

In [ ]:
def load_data():
    df_train = pd.read_csv("train.csv", index_col = "Id")
    df_test = pd.read_csv("test.csv", index_col = "Id")

    df = pd.concat([df_train, df_test])
    df = clean(df)
    df = encode(df)
    df = impute(df)

    df_train = df.loc[df_train.index, :]
    df_test = df.loc[df_test.index, :]
    return df_train, df_test


In [ ]:

df = pd.read_csv("train.csv", index_col="Id")

df.Exterior2nd.unique()

In [ ]:
def clean(df):
    df["Exterior2nd"] = df["Exterior2nd"].replace({"Brk Cmn": "BrkComm"})
    # Some values of GarageYrBlt are corrupt, so we'll replace them
    # with the year the house was built
    df["GarageYrBlt"] = df["GarageYrBlt"].where(df.GarageYrBlt <= 2010, df.YearBuilt)
    # Names beginning with numbers are awkward to work with
    df.rename(columns={
        "1stFlrSF": "FirstFlrSF",
        "2ndFlrSF": "SecondFlrSF",
        "3SsnPorch": "Threeseasonporch",
    }, inplace=True,
    )
    return df

In [ ]:

# The numeric features are already encoded correctly (`float` for
# continuous, `int` for discrete), but the categoricals we'll need to
# do ourselves. Note in particular, that the `MSSubClass` feature is
# read as an `int` type, but is actually a (nominative) categorical.

# The nominative (unordered) categorical features
features_nom = ["MSSubClass", "MSZoning", "Street", "Alley", "LandContour", "LotConfig", "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", "Heating", "CentralAir", "GarageType", "MiscFeature", "SaleType", "SaleCondition"]


# The ordinal (ordered) categorical features 

# Pandas calls the categories "levels"
five_levels = ["Po", "Fa", "TA", "Gd", "Ex"]
ten_levels = list(range(10))

ordered_levels = {
    "OverallQual": ten_levels,
    "OverallCond": ten_levels,
    "ExterQual": five_levels,
    "ExterCond": five_levels,
    "BsmtQual": five_levels,
    "BsmtCond": five_levels,
    "HeatingQC": five_levels,
    "KitchenQual": five_levels,
    "FireplaceQu": five_levels,
    "GarageQual": five_levels,
    "GarageCond": five_levels,
    "PoolQC": five_levels,
    "LotShape": ["Reg", "IR1", "IR2", "IR3"],
    "LandSlope": ["Sev", "Mod", "Gtl"],
    "BsmtExposure": ["No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "Functional": ["Sal", "Sev", "Maj1", "Maj2", "Mod", "Min2", "Min1", "Typ"],
    "GarageFinish": ["Unf", "RFn", "Fin"],
    "PavedDrive": ["N", "P", "Y"],
    "Utilities": ["NoSeWa", "NoSewr", "AllPub"],
    "CentralAir": ["N", "Y"],
    "Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
    "Fence": ["MnWw", "GdWo", "MnPrv", "GdPrv"],
}

# Add a None level for missing values
ordered_levels = {key: ["None"] + value for key, value in
                  ordered_levels.items()}


def encode(df):
    # Nominal categories
    for name in features_nom:
        df[name] = df[name].astype("category")
        # Add a None category for missing values
        if "None" not in df[name].cat.categories:
            df[name] = df[name].cat.add_categories("None")
    # Ordinal categories
    for name, levels in ordered_levels.items():
        df[name] = df[name].astype(CategoricalDtype(levels,
                                                    ordered=True))
    return df


In [ ]:
def impute(df):
    for name in df.select_dtypes("number"):
        df[name] = df[name].fillna(0)
    for name in df.select_dtypes("category"):
        df[name] = df[name].fillna("None")
    return df


In [ ]:
df_train, df_test = load_data()

In [ ]:
# Peek at the values
display(df_train)
display(df_test)

# Display information about dtypes and missing values
display(df_train.info())
display(df_test.info())

In [3]:

def score_dataset(X, y, model=XGBRegressor()):
    X = X.copy()

    for colname in X.select_dtypes(include=["object", "string", "category"]):
        X[colname], _ = X[colname].factorize()

    log_y = np.log(y)

    score = cross_val_score(
        model,
        X,
        log_y,
        cv=5,
        scoring="neg_mean_squared_error"
    )

    return np.sqrt(-score.mean())


In [ ]:
X = df_train.copy()
y = X.pop("SalePrice")

baseline_score = score_dataset(X, y)
print(f"Baseline score: {baseline_score:.5f} RMSLE")

In [ ]:
#X = pd.get_dummies(X,columns=features_nom,dtype=int)

In [7]:
def make_mi_scores(X, y):
    X = X.copy()

    for colname in X.select_dtypes(include=["object", "string", "category"]):
        X[colname], _ = X[colname].factorize()

    discrete_features = [
        pd.api.types.is_integer_dtype(X[col]) and X[col].nunique() < 20
        for col in X.columns
    ]

    mi_scores = mutual_info_regression(
        X,
        y,
        discrete_features=discrete_features,
        random_state=0
    )

    mi_scores = pd.Series(
        mi_scores,
        name="MI Scores",
        index=X.columns
    )

    return mi_scores.sort_values(ascending=False)

In [ ]:
mi_scores = make_mi_scores(X, y)

In [ ]:
def plot_mi_scores(scores):
    scores = scores.sort_values(ascending=True)
    width = np.arange(len(scores))
    ticks = list(scores.index)
    plt.barh(width, scores)
    plt.yticks(width, ticks)
    plt.title("Mutual Information Scores")




In [ ]:
mi_scores["BldgType"]

In [ ]:
X["BldgType"]

In [ ]:
def drop_uninformative(df, mi_scores):
    features = mi_scores[mi_scores > 0].index.tolist()

    if "MiscFeature" in df.columns and "MiscFeature" not in features:
        features.append("MiscFeature")

    return df[features]

In [ ]:
print(X.columns.equals(mi_scores.index))

In [ ]:
print(X.columns)
print(mi_scores.index)

In [ ]:
mi_scores["MiscFeature"]

In [ ]:
X = df_train.copy()
y = X.pop("SalePrice")
X = drop_uninformative(X, mi_scores)

df_test = drop_uninformative(df_test,mi_scores)

score_dataset(X, y)

In [ ]:
def label_encode(df):
    X = df.copy()
    for colname in X.select_dtypes(["category"]):
        X[colname] = X[colname].cat.codes
    return X

X = label_encode(X)
df_test = label_encode(df_test)

In [ ]:

def mathematical_transforms(df):
    X = pd.DataFrame()  # dataframe to hold new features
    X["LivLotRatio"] = df.GrLivArea / df.LotArea
    X["Spaciousness"] = (df.FirstFlrSF + df.SecondFlrSF) / df.TotRmsAbvGrd
    # This feature ended up not helping performance
    # X["TotalOutsideSF"] = \
    #     df.WoodDeckSF + df.OpenPorchSF + df.EnclosedPorch + \
    #     df.Threeseasonporch + df.ScreenPorch
    return X


def interactions(df):
    X = pd.get_dummies(df.BldgType, prefix="Bldg")
    X = X.mul(df.GrLivArea, axis=0)
    return X


def counts(df):
    X = pd.DataFrame()
    X["PorchTypes"] = df[[
        "WoodDeckSF",
        "OpenPorchSF",
        "EnclosedPorch",
        "Threeseasonporch",
        "ScreenPorch",
    ]].gt(0.0).sum(axis=1)
    return X



def group_transforms(df):
    X = pd.DataFrame()
    X["MedNhbdArea"] = df.groupby("Neighborhood")["GrLivArea"].transform("median")
    return X

def add_features(df):
    X = df.copy()

    utilities_map = {"AllPub": 3, "NoSewr": 2, "NoSeWa": 1, "ELO": 0}
    landslope_map = {"Gtl": 3, "Mod": 2, "Sev": 1}
    exterqual_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0}
    extercond_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0}
    bsmtqual_map = {"Ex": 100, "Gd": 95, "TA": 85, "Fa": 75, "Po": 65, "NA": 0}
    bsmtcond_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "NA": 0}
    bsmtexposure_map = {"Gd": 4, "Av": 3, "Mn": 2, "No": 1, "NA": 0}
    bsmtfintype_map = {"GLQ": 6, "ALQ": 5, "BLQ": 4, "Rec": 3, "LwQ": 2, "Unf": 1, "NA": 0}
    heatingqc_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0}
    centralair_map = {"Y": 1, "N": 0}
    kitchenqual_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0}
    functional_map = {"Typ": 7, "Min1": 6, "Min2": 5, "Mod": 4, "Maj1": 3, "Maj2": 2, "Sev": 1, "Sal": 0}
    fireplacequ_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "NA": 0}
    garagefinish_map = {"Fin": 3, "RFn": 2, "Unf": 1, "NA": 0}
    garagequal_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "NA": 0}
    garagecond_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "NA": 0}
    paveddrive_map = {"Y": 2, "P": 1, "N": 0}
    poolqc_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "NA": 0}
    fence_map = {"GdPrv": 4, "MnPrv": 3, "GdWo": 2, "MnWw": 1, "NA": 0}

    X["Utilities"] = X["Utilities"].fillna("ELO").map(utilities_map)
    X["LandSlope"] = X["LandSlope"].map(landslope_map)
    X["ExterQual"] = X["ExterQual"].map(exterqual_map)
    X["ExterCond"] = X["ExterCond"].map(extercond_map)
    X["BsmtQual"] = X["BsmtQual"].fillna("NA").map(bsmtqual_map)
    X["BsmtCond"] = X["BsmtCond"].fillna("NA").map(bsmtcond_map)
    X["BsmtExposure"] = X["BsmtExposure"].fillna("NA").map(bsmtexposure_map)
    X["BsmtFinType1"] = X["BsmtFinType1"].fillna("NA").map(bsmtfintype_map)
    X["BsmtFinType2"] = X["BsmtFinType2"].fillna("NA").map(bsmtfintype_map)
    X["HeatingQC"] = X["HeatingQC"].map(heatingqc_map)
    X["CentralAir"] = X["CentralAir"].fillna("N").map(centralair_map)
    X["KitchenQual"] = X["KitchenQual"].fillna("TA").map(kitchenqual_map)
    X["Functional"] = X["Functional"].fillna("Typ").map(functional_map)
    X["FireplaceQu"] = X["FireplaceQu"].fillna("NA").map(fireplacequ_map)
    X["GarageFinish"] = X["GarageFinish"].fillna("NA").map(garagefinish_map)
    X["GarageQual"] = X["GarageQual"].fillna("NA").map(garagequal_map)
    X["GarageCond"] = X["GarageCond"].fillna("NA").map(garagecond_map)
    X["PavedDrive"] = X["PavedDrive"].map(paveddrive_map)
    X["Fence"] = X["Fence"].fillna("NA").map(fence_map)

    rail_conditions = ["RRNn", "RRAn", "RRNe", "RRAe"]

    X["Lotareafrontage"] = X["LotArea"] * X["LotFrontage"]
    X["Lotfrontagearea"] = X["LotArea"] / X["LotFrontage"].replace(0, 1)
    X["LotShape"] = X["LotShape"].fillna("U")
    X["LandContour"] = X["LandContour"].fillna("U")
    X["Combinedcontourshape"] = X["LotShape"].astype(str) + "_" + X["LandContour"].astype(str)
    X["nearrailroad"] = (X["Condition1"].isin(rail_conditions) | X["Condition2"].isin(rail_conditions)).astype(int)
    X["CombinedCondition"] = X["Condition1"].astype(str) + "_" + X["Condition2"].astype(str)
    X["hasfinishedsecondlevel"] = X["HouseStyle"].isin(["1.5Fin", "2Story", "2.5Fin"]).astype(int)
    X["OverallQualcond"] = (2 * X["OverallQual"] * X["OverallCond"]) / (X["OverallQual"] + X["OverallCond"])
    X["Yearstoremodel"] = X["YearRemodAdd"] - X["YearBuilt"]
    X["Exterior"] = X["Exterior1st"].astype(str) + "_" + X["Exterior2nd"].astype(str)
    X["hasmasonyveer"] = (X["MasVnrArea"].fillna(0) != 0).astype(int)
    X["hasbasement"] = (X["BsmtQual"] != 0).astype(int)
    X["Finishedbasement"] = X["TotalBsmtSF"] - X["BsmtUnfSF"]
    X["hassecondfloor"] = (X["SecondFlrSF"] != 0).astype(int)
    X["haslowqualfinish"] = (X["LowQualFinSF"] != 0).astype(int)
    X["TotalBathrooms"] = X["FullBath"] + 0.5 * X["HalfBath"] + X["BsmtFullBath"].fillna(0) + 0.5 * X["BsmtHalfBath"].fillna(0)
    X["hasfireplace"] = (X["Fireplaces"] != 0).astype(int)
    X["hasgarage"] = (X["GarageCars"].fillna(0) != 0).astype(int)
    X["GarageQualcond"] = (2 * X["GarageQual"] * X["GarageCond"]) / (X["GarageQual"] + X["GarageCond"]).replace(0, 1)
    X["haswooddeck"] = (X["WoodDeckSF"] != 0).astype(int)
    X["hasopenporch"] = (X["OpenPorchSF"] != 0).astype(int)
    X["hasenclosedporch"] = (X["EnclosedPorch"] != 0).astype(int)
    X["has3seasonporch"] = (X["Threeseasonporch"] != 0).astype(int)
    X["hasscreenporch"] = (X["ScreenPorch"] != 0).astype(int)
    X["haspool"] = (X["PoolArea"] != 0).astype(int)
    X["hasfence"] = (X["Fence"] != 0).astype(int)
    X["haselevator"] = (X["MiscFeature"] == "Elev").astype(int)
    X["hassecondgarage"] = (X["MiscFeature"] == "Gar2").astype(int)
    X["hasshed"] = (X["MiscFeature"] == "Shed").astype(int)
    X["hastenniscourt"] = (X["MiscFeature"] == "TenC").astype(int)

    X.drop(["Condition1", "Condition2", "YearRemodAdd", "Exterior1st", "Exterior2nd"], axis=1, inplace=True)

    return X

In [ ]:
X["BldgType"]

In [ ]:
X = add_features(X)

new_features = pd.concat([
    mathematical_transforms(X),
    interactions(X),
    counts(X),
    group_transforms(X)
], axis=1)

X = X.join(new_features)

df_test = add_features(df_test)

new_features = pd.concat([
    mathematical_transforms(df_test),
    interactions(df_test),
    counts(df_test),
    group_transforms(df_test)
], axis=1)

df_test = df_test.join(new_features)

In [ ]:
score_dataset(X, y)

In [ ]:
X.columns[X.isna().any()]

In [ ]:
X.isna().sum()[X.isna().sum() > 0]

In [ ]:
mi_scores = make_mi_scores(X, y)

In [4]:
import numpy as np
import pandas as pd


def feature_engineering(df, neighborhood_medians=None):
    X = df.copy()

    # --------------------------------------------------
    # Fix / fill values needed for feature engineering
    # --------------------------------------------------

    X["GarageYrBlt"] = X["GarageYrBlt"].where(X["GarageYrBlt"] <= 2010, X["YearBuilt"])

    categorical_none = [
        "Alley", "MasVnrType", "BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2", "FireplaceQu", "GarageType",
        "GarageFinish", "GarageQual", "GarageCond", "PoolQC", "Fence",
        "MiscFeature"
    ]

    for col in categorical_none:
        X[col] = X[col].fillna("None")

    categorical_mode = [
        "MSZoning", "Utilities", "Exterior1st", "Exterior2nd",
        "Electrical", "KitchenQual", "Functional", "SaleType"
    ]

    for col in categorical_mode:
        X[col] = X[col].fillna(X[col].mode()[0])

    numeric_fill_zero = [
        "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
        "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath",
        "GarageCars", "GarageArea", "GarageYrBlt"
    ]

    for col in numeric_fill_zero:
        X[col] = X[col].fillna(0)

    X["LotFrontage"] = X["LotFrontage"].fillna(X["LotFrontage"].median())


    # --------------------------------------------------
    # Ordinal mappings
    # --------------------------------------------------

    utilities_map = {"AllPub": 3, "NoSewr": 2, "NoSeWa": 1, "ELO": 0}
    landslope_map = {"Gtl": 3, "Mod": 2, "Sev": 1}
    qual_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0, "None": 0}
    bsmtqual_map = {"Ex": 100, "Gd": 95, "TA": 85, "Fa": 75, "Po": 65, "None": 0}
    bsmtcond_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0}
    bsmtexposure_map = {"Gd": 4, "Av": 3, "Mn": 2, "No": 1, "None": 0}
    bsmtfintype_map = {"GLQ": 6, "ALQ": 5, "BLQ": 4, "Rec": 3, "LwQ": 2, "Unf": 1, "None": 0}
    centralair_map = {"Y": 1, "N": 0}
    functional_map = {"Typ": 7, "Min1": 6, "Min2": 5, "Mod": 4, "Maj1": 3, "Maj2": 2, "Sev": 1, "Sal": 0}
    fireplacequ_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0}
    garagefinish_map = {"Fin": 3, "RFn": 2, "Unf": 1, "None": 0}
    garagequal_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0}
    paveddrive_map = {"Y": 2, "P": 1, "N": 0}
    poolqc_map = {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "None": 0}
    fence_map = {"GdPrv": 4, "MnPrv": 3, "GdWo": 2, "MnWw": 1, "None": 0}

    X["Utilities"] = X["Utilities"].map(utilities_map)
    X["LandSlope"] = X["LandSlope"].map(landslope_map)
    X["ExterQual"] = X["ExterQual"].map(qual_map)
    X["ExterCond"] = X["ExterCond"].map(qual_map)
    X["BsmtQual"] = X["BsmtQual"].map(bsmtqual_map)
    X["BsmtCond"] = X["BsmtCond"].map(bsmtcond_map)
    X["BsmtExposure"] = X["BsmtExposure"].map(bsmtexposure_map)
    X["BsmtFinType1"] = X["BsmtFinType1"].map(bsmtfintype_map)
    X["BsmtFinType2"] = X["BsmtFinType2"].map(bsmtfintype_map)
    X["HeatingQC"] = X["HeatingQC"].map(qual_map)
    X["CentralAir"] = X["CentralAir"].map(centralair_map)
    X["KitchenQual"] = X["KitchenQual"].map(qual_map)
    X["Functional"] = X["Functional"].map(functional_map)
    X["FireplaceQu"] = X["FireplaceQu"].map(fireplacequ_map)
    X["GarageFinish"] = X["GarageFinish"].map(garagefinish_map)
    X["GarageQual"] = X["GarageQual"].map(garagequal_map)
    X["GarageCond"] = X["GarageCond"].map(garagequal_map)
    X["PavedDrive"] = X["PavedDrive"].map(paveddrive_map)
    X["PoolQC"] = X["PoolQC"].map(poolqc_map)
    X["Fence"] = X["Fence"].map(fence_map)


    # --------------------------------------------------
    # Mathematical features
    # --------------------------------------------------

    X["LivLotRatio"] = X["GrLivArea"] / X["LotArea"].replace(0, 1)
    X["Spaciousness"] = (X["1stFlrSF"] + X["2ndFlrSF"]) / X["TotRmsAbvGrd"].replace(0, 1)
    X["Lotareafrontage"] = X["LotArea"] * X["LotFrontage"]
    X["Lotfrontagearea"] = X["LotArea"] / X["LotFrontage"].replace(0, 1)


    # --------------------------------------------------
    # Interaction / combined categorical features
    # --------------------------------------------------

    X["LotShape"] = X["LotShape"].fillna("U")
    X["LandContour"] = X["LandContour"].fillna("U")
    X["Combinedcontourshape"] = X["LotShape"].astype(str) + "_" + X["LandContour"].astype(str)

    rail_conditions = ["RRNn", "RRAn", "RRNe", "RRAe"]
    X["nearrailroad"] = (X["Condition1"].isin(rail_conditions) | X["Condition2"].isin(rail_conditions)).astype(int)
    X["CombinedCondition"] = X["Condition1"].astype(str) + "_" + X["Condition2"].astype(str)

    X["Exterior"] = X["Exterior1st"].astype(str) + "_" + X["Exterior2nd"].astype(str)

    X["hasfinishedsecondlevel"] = X["HouseStyle"].isin(["1.5Fin", "2Story", "2.5Fin"]).astype(int)
    X["OverallQualcond"] = (2 * X["OverallQual"] * X["OverallCond"]) / (X["OverallQual"] + X["OverallCond"]).replace(0, 1)


    # --------------------------------------------------
    # Age features
    # --------------------------------------------------

    X["Ageatsale"] = X["YrSold"] - X["YearBuilt"]
    X["Yearstoremodel"] = X["YearRemodAdd"] - X["YearBuilt"]
    X["Garageageatsale"] = (X["YrSold"] - X["GarageYrBlt"]).clip(lower=0)


    # --------------------------------------------------
    # Presence / count features
    # --------------------------------------------------

    X["hasmasonyveer"] = (X["MasVnrArea"] != 0).astype(int)
    X["hasbasement"] = (X["TotalBsmtSF"] != 0).astype(int)
    X["Finishedbasement"] = X["TotalBsmtSF"] - X["BsmtUnfSF"]
    X["hassecondfloor"] = (X["2ndFlrSF"] != 0).astype(int)
    X["haslowqualfinish"] = (X["LowQualFinSF"] != 0).astype(int)
    X["TotalBathrooms"] = X["FullBath"] + 0.5 * X["HalfBath"] + X["BsmtFullBath"] + 0.5 * X["BsmtHalfBath"]
    X["hasfireplace"] = (X["Fireplaces"] != 0).astype(int)
    X["hasgarage"] = (X["GarageCars"] != 0).astype(int)
    X["GarageQualcond"] = (2 * X["GarageQual"] * X["GarageCond"]) / (X["GarageQual"] + X["GarageCond"]).replace(0, 1)
    X["haswooddeck"] = (X["WoodDeckSF"] != 0).astype(int)
    X["hasopenporch"] = (X["OpenPorchSF"] != 0).astype(int)
    X["hasenclosedporch"] = (X["EnclosedPorch"] != 0).astype(int)
    X["has3seasonporch"] = (X["3SsnPorch"] != 0).astype(int)
    X["hasscreenporch"] = (X["ScreenPorch"] != 0).astype(int)
    X["haspool"] = (X["PoolArea"] != 0).astype(int)
    X["hasfence"] = (X["Fence"] != 0).astype(int)
    X["haselevator"] = (X["MiscFeature"] == "Elev").astype(int)
    X["hassecondgarage"] = (X["MiscFeature"] == "Gar2").astype(int)
    X["hasshed"] = (X["MiscFeature"] == "Shed").astype(int)
    X["hastenniscourt"] = (X["MiscFeature"] == "TenC").astype(int)

    X["PorchTypes"] = X[["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]].gt(0).sum(axis=1)


    # --------------------------------------------------
    # Neighborhood feature
    # --------------------------------------------------

    if neighborhood_medians is None:
        neighborhood_medians = X.groupby("Neighborhood")["GrLivArea"].median()

    X["MedNhbdArea"] = X["Neighborhood"].map(neighborhood_medians)
    X["MedNhbdArea"] = X["MedNhbdArea"].fillna(neighborhood_medians.median())


    # --------------------------------------------------
    # Treat MSSubClass as categorical, not continuous
    # --------------------------------------------------

    X["MSSubClass"] = X["MSSubClass"].astype(str)


    # --------------------------------------------------
    # Remove columns replaced by engineered versions
    # --------------------------------------------------

    X.drop(["Condition1", "Condition2", "Exterior1st", "Exterior2nd", "YearRemodAdd"], axis=1, inplace=True)


    # --------------------------------------------------
    # Final missing values
    # --------------------------------------------------

    for col in X.select_dtypes(include=["object", "string", "category"]).columns:
        X[col] = X[col].fillna("None")

    for col in X.select_dtypes(include=np.number).columns:
        X[col] = X[col].fillna(X[col].median())

    return X, neighborhood_medians


# ======================================================
# APPLY TO TRAIN + TEST
# ======================================================

X = df_train.copy()
y = X.pop("SalePrice")

test_ids = df_test["Id"].copy()

X = X.drop(columns=["Id"])
df_test_processed = df_test.drop(columns=["Id"]).copy()

X, neighborhood_medians = feature_engineering(X)
df_test_processed, _ = feature_engineering(df_test_processed, neighborhood_medians)


# ======================================================
# BUILDING-TYPE × LIVING-AREA INTERACTIONS
# ======================================================

train_bldg = pd.get_dummies(X["BldgType"], prefix="Bldg", dtype=int).mul(X["GrLivArea"], axis=0)
test_bldg = pd.get_dummies(df_test_processed["BldgType"], prefix="Bldg", dtype=int).mul(df_test_processed["GrLivArea"], axis=0)

train_bldg, test_bldg = train_bldg.align(test_bldg, join="outer", axis=1, fill_value=0)

X = X.join(train_bldg)
df_test_processed = df_test_processed.join(test_bldg)


# ======================================================
# LABEL ENCODE ALL REMAINING STRING FEATURES
# SAME CODES FOR TRAIN + TEST
# ======================================================

categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns

for col in categorical_columns:
    combined = pd.concat([X[col], df_test_processed[col]], axis=0, ignore_index=True).astype(str)
    codes, _ = pd.factorize(combined)

    X[col] = codes[:len(X)]
    df_test_processed[col] = codes[len(X):]


# ======================================================
# FINAL SAFETY CHECK
# ======================================================

X = X.replace([np.inf, -np.inf], np.nan)
df_test_processed = df_test_processed.replace([np.inf, -np.inf], np.nan)

X = X.fillna(X.median(numeric_only=True))
df_test_processed = df_test_processed.fillna(X.median(numeric_only=True))

print("Train shape:", X.shape)
print("Test shape:", df_test_processed.shape)
print("Train NaNs:", X.isna().sum().sum())
print("Test NaNs:", df_test_processed.isna().sum().sum())
print("Non-numeric train columns:", X.select_dtypes(exclude=["number", "bool"]).columns.tolist())

Train shape: (1460, 114)
Test shape: (1459, 114)
Train NaNs: 0
Test NaNs: 0
Non-numeric train columns: []


In [8]:
mi_scores = make_mi_scores(X, y)

In [9]:
def drop_uninformative(df, mi_scores):
    count = (mi_scores == 0).count()
    features = mi_scores[mi_scores > 0.01].index.tolist()
    
    return count,df[features]

count1, X = drop_uninformative(X, mi_scores)
count2, df_test_processed= drop_uninformative(df_test_processed, mi_scores)

print(count1)
print(count2)

114
114


In [ ]:
X

In [ ]:
mi_scores = make_mi_scores(X, y)

In [ ]:
mi_scores

In [10]:
X["QualGrLiv"] = X["OverallQual"] * X["GrLivArea"]
X["QualNeighborhoodArea"] = X["OverallQual"] * X["MedNhbdArea"]
X["GrLivVsNeighborhood"] = X["GrLivArea"] / X["MedNhbdArea"].replace(0, 1)

df_test_processed["QualGrLiv"] = df_test_processed["OverallQual"] * df_test_processed["GrLivArea"]
df_test_processed["QualNeighborhoodArea"] = df_test_processed["OverallQual"] * df_test_processed["MedNhbdArea"]
df_test_processed["GrLivVsNeighborhood"] = df_test_processed["GrLivArea"] / df_test_processed["MedNhbdArea"].replace(0, 1)

In [ ]:
mi_scores = make_mi_scores(X, y)
mi_scores

In [ ]:
score_dataset(X,y)

In [ ]:
print("All features:", score_dataset(X, y))

for threshold in [0, 0.005, 0.01, 0.02, 0.05]:
    features = mi_scores[mi_scores > threshold].index
    print(threshold, len(features), score_dataset(X[features], y))

In [11]:
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.001,
    reg_lambda=1,
    objective="reg:squarederror",
    random_state=42
)

score_dataset(X, y, model)

np.float64(0.11897930398281543)

In [12]:
import numpy as np
import optuna

from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Log-transform target
y_log = np.log1p(y)

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 1000, 6000),
        "max_depth": trial.suggest_int("max_depth", 2, 8),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.3, log=True
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight", 0.05, 5, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.5, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.5, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 10
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 0.02, 10, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.02, 100, log=True
        )
    }

    model = XGBRegressor(
        **params,
        objective="reg:squarederror",
        random_state=42
    )

    score = cross_val_score(
        model,
        X,
        y_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    return -score.mean()


study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=30
)

print("Best log-RMSE:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-08-31 16:25:02,146] A new study created in memory with name: no-name-b9756ba1-f926-4475-9b51-a91ba34407b8
[I 2026-08-31 16:25:04,672] Trial 0 finished with value: 0.17115730821407613 and parameters: {'n_estimators': 2304, 'max_depth': 5, 'learning_rate': 0.08400164514740352, 'min_child_weight': 0.1635920180005314, 'subsample': 0.6528525642006062, 'colsample_bytree': 0.5307124059184429, 'gamma': 2.2429048980691446, 'reg_alpha': 2.006293268189478, 'reg_lambda': 0.06418386163672712}. Best is trial 0 with value: 0.17115730821407613.
[I 2026-08-31 16:25:07,296] Trial 1 finished with value: 0.14684602036843486 and parameters: {'n_estimators': 4324, 'max_depth': 8, 'learning_rate': 0.022268945439272433, 'min_child_weight': 2.595451854818507, 'subsample': 0.7395299802617931, 'colsample_bytree': 0.9403295710487205, 'gamma': 0.9279274960223183, 'reg_alpha': 0.18739464235174388, 'reg_lambda': 0.11919930043829767}. Best is trial 1 with value: 0.14684602036843486.
[I 2026-08-31 16:25:09,384

Best log-RMSE: 0.13243198513279758
Best parameters: {'n_estimators': 5906, 'max_depth': 6, 'learning_rate': 0.026424698776337366, 'min_child_weight': 3.5374788644037722, 'subsample': 0.7999567065930824, 'colsample_bytree': 0.9033791620498391, 'gamma': 0.007886975332884495, 'reg_alpha': 0.35102456908101554, 'reg_lambda': 0.021304270009066383}


In [13]:
XGB_best_model_previous = XGBRegressor(
    **study.best_params,
    objective="reg:squarederror",
    random_state=42
)

XGB_best_model_previous.fit(X, np.log1p(y))

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9033791620498391
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [14]:
pred_log = XGB_best_model_previous.predict(df_test_processed)

predictions = np.expm1(pred_log)

In [16]:
score_dataset(X,y,XGB_best_model_previous)

np.float64(0.12330975038426696)

In [15]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": predictions
})

submission.to_csv("submissionengineered2.csv", index=False)

In [17]:
# Quality × size
X["QualGrLiv"] = X["OverallQual"] * X["GrLivArea"]
X["QualTotalBsmt"] = X["OverallQual"] * X["TotalBsmtSF"]
X["QualGarageArea"] = X["OverallQual"] * X["GarageArea"]

# Size combinations
X["TotalSF"] = X["1stFlrSF"] + X["2ndFlrSF"] + X["TotalBsmtSF"]

# Age / quality
X["AgeQual"] = X["Ageatsale"] * X["OverallQual"]

# Neighborhood-relative
X["GrLivVsNeighborhood"] = X["GrLivArea"] / X["MedNhbdArea"].replace(0, 1)
X["QualNeighborhood"] = X["OverallQual"] * X["MedNhbdArea"]

# Garage
X["GarageCarArea"] = X["GarageCars"] * X["GarageArea"]

# Bathrooms / rooms
X["BathPerRoom"] = X["TotalBathrooms"] / X["TotRmsAbvGrd"].replace(0, 1)
X["AreaPerRoom"] = X["GrLivArea"] / X["TotRmsAbvGrd"].replace(0, 1)

In [18]:
df_test_processed["QualGrLiv"] = df_test_processed["OverallQual"] * df_test_processed["GrLivArea"]
df_test_processed["QualTotalBsmt"] = df_test_processed["OverallQual"] * df_test_processed["TotalBsmtSF"]
df_test_processed["QualGarageArea"] = df_test_processed["OverallQual"] * df_test_processed["GarageArea"]

df_test_processed["TotalSF"] = df_test_processed["1stFlrSF"] + df_test_processed["2ndFlrSF"] + df_test_processed["TotalBsmtSF"]

df_test_processed["AgeQual"] = df_test_processed["Ageatsale"] * df_test_processed["OverallQual"]

df_test_processed["GrLivVsNeighborhood"] = df_test_processed["GrLivArea"] / df_test_processed["MedNhbdArea"].replace(0, 1)
df_test_processed["QualNeighborhood"] = df_test_processed["OverallQual"] * df_test_processed["MedNhbdArea"]

df_test_processed["GarageCarArea"] = df_test_processed["GarageCars"] * df_test_processed["GarageArea"]

df_test_processed["BathPerRoom"] = df_test_processed["TotalBathrooms"] / df_test_processed["TotRmsAbvGrd"].replace(0, 1)
df_test_processed["AreaPerRoom"] = df_test_processed["GrLivArea"] / df_test_processed["TotRmsAbvGrd"].replace(0, 1)

In [ ]:
mi_scores = make_mi_scores(X,y)

In [ ]:
mi_scores

In [19]:
import optuna 
from sklearn.model_selection import cross_val_score, KFold

from sklearn.model_selection import StratifiedKFold
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 2000,6000)
    max_depth = trial.suggest_int("max_depth",2,8)
    learning_rate = trial.suggest_float("learning_rate",0.005,0.3,log=True)
    min_child_weight = trial.suggest_float("min_child_weight",0.05,5,log=True)

    subsample = trial.suggest_float("subsample",0.5,1.0)

    colsample_bytree = trial.suggest_float("colsample_bytree",0.5,1.0)

    gamma = trial.suggest_float("gamma",0,10)

    reg_alpha = trial.suggest_float("reg_alpha",0.02,10,log=True)
    reg_lambda = trial.suggest_float("reg_lambda",0.02,100,log=True)

    model = XGBRegressor(
        n_estimators= n_estimators,
        max_depth = max_depth,
        learning_rate = learning_rate,
        min_child_weight = min_child_weight,
        subsample = subsample,
        colsample_bytree = colsample_bytree,
        gamma = gamma,
        objective="reg:squarederror",
        reg_alpha = reg_alpha,
        reg_lambda = reg_lambda
    )

        
    score = cross_val_score(model, X, y, cv = cv, 
                            scoring ="neg_root_mean_squared_error")
    return -score.mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials = 50)

print("Best CV score:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-08-31 16:28:42,288] A new study created in memory with name: no-name-8dc2328e-f626-419e-b674-edc4ef813b04
[I 2026-08-31 16:28:50,992] Trial 0 finished with value: 28483.289453125 and parameters: {'n_estimators': 2366, 'max_depth': 5, 'learning_rate': 0.13041900912252974, 'min_child_weight': 0.5239157606448943, 'subsample': 0.8353573132615602, 'colsample_bytree': 0.7476113910299564, 'gamma': 8.27410113509614, 'reg_alpha': 0.8253764987020127, 'reg_lambda': 50.3081962828308}. Best is trial 0 with value: 28483.289453125.
[I 2026-08-31 16:29:13,376] Trial 1 finished with value: 27950.79765625 and parameters: {'n_estimators': 3799, 'max_depth': 6, 'learning_rate': 0.017372033103555853, 'min_child_weight': 0.6451980165559607, 'subsample': 0.7343932737645635, 'colsample_bytree': 0.7018863566835418, 'gamma': 4.757569929900787, 'reg_alpha': 0.2700173410579443, 'reg_lambda': 0.4152901039008123}. Best is trial 1 with value: 27950.79765625.
[I 2026-08-31 16:29:43,527] Trial 2 finished with 

KeyboardInterrupt: 

In [21]:
import numpy as np
import optuna

from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS_TREE = 30
N_TRIALS_RIDGE = 20
N_TRIALS_WEIGHTS = 100

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
y_log = np.log1p(np.asarray(y))

def cv_rmse(model):
    scores = cross_val_score(model, X, y_log, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=1)
    return -scores.mean()

def objective_xgb(trial):
    model = XGBRegressor(n_estimators=trial.suggest_int("n_estimators", 500, 5000), learning_rate=trial.suggest_float("learning_rate", 0.005, 0.15, log=True), max_depth=trial.suggest_int("max_depth", 2, 8), min_child_weight=trial.suggest_float("min_child_weight", 0.1, 20, log=True), subsample=trial.suggest_float("subsample", 0.5, 1.0), colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0), gamma=trial.suggest_float("gamma", 0.0, 5.0), reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True), reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 100, log=True), objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1)
    return cv_rmse(model)

def objective_cat(trial):
    model = CatBoostRegressor(iterations=trial.suggest_int("iterations", 500, 5000), learning_rate=trial.suggest_float("learning_rate", 0.005, 0.15, log=True), depth=trial.suggest_int("depth", 4, 10), l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 0.1, 30, log=True), random_strength=trial.suggest_float("random_strength", 1e-3, 10, log=True), bagging_temperature=trial.suggest_float("bagging_temperature", 0, 10), loss_function="RMSE", verbose=False, random_seed=RANDOM_STATE, thread_count=-1)
    return cv_rmse(model)

def objective_lgbm(trial):
    model = LGBMRegressor(n_estimators=trial.suggest_int("n_estimators", 500, 5000), learning_rate=trial.suggest_float("learning_rate", 0.005, 0.15, log=True), num_leaves=trial.suggest_int("num_leaves", 10, 150), max_depth=trial.suggest_int("max_depth", 3, 12), min_child_samples=trial.suggest_int("min_child_samples", 5, 100), subsample=trial.suggest_float("subsample", 0.5, 1.0), colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0), reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True), reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 100, log=True), random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
    return cv_rmse(model)

def objective_extra(trial):
    model = ExtraTreesRegressor(n_estimators=trial.suggest_int("n_estimators", 300, 1500), max_depth=trial.suggest_int("max_depth", 5, 40), min_samples_split=trial.suggest_int("min_samples_split", 2, 20), min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10), max_features=trial.suggest_float("max_features", 0.4, 1.0), random_state=RANDOM_STATE, n_jobs=-1)
    return cv_rmse(model)

def objective_rf(trial):
    model = RandomForestRegressor(n_estimators=trial.suggest_int("n_estimators", 300, 1500), max_depth=trial.suggest_int("max_depth", 5, 40), min_samples_split=trial.suggest_int("min_samples_split", 2, 20), min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10), max_features=trial.suggest_float("max_features", 0.4, 1.0), random_state=RANDOM_STATE, n_jobs=-1)
    return cv_rmse(model)

def objective_ridge(trial):
    model = make_pipeline(RobustScaler(), Ridge(alpha=trial.suggest_float("alpha", 1e-4, 1000, log=True)))
    return cv_rmse(model)

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_TREE)

study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(objective_cat, n_trials=N_TRIALS_TREE)

study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS_TREE)

study_extra = optuna.create_study(direction="minimize")
study_extra.optimize(objective_extra, n_trials=N_TRIALS_TREE)

study_rf = optuna.create_study(direction="minimize")
study_rf.optimize(objective_rf, n_trials=N_TRIALS_TREE)

study_ridge = optuna.create_study(direction="minimize")
study_ridge.optimize(objective_ridge, n_trials=N_TRIALS_RIDGE)

print("XGB:", study_xgb.best_value, study_xgb.best_params)
print("CAT:", study_cat.best_value, study_cat.best_params)
print("LGBM:", study_lgbm.best_value, study_lgbm.best_params)
print("EXTRA:", study_extra.best_value, study_extra.best_params)
print("RF:", study_rf.best_value, study_rf.best_params)
print("RIDGE:", study_ridge.best_value, study_ridge.best_params)

xgb = XGBRegressor(**study_xgb.best_params, objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1)
cat = CatBoostRegressor(**study_cat.best_params, loss_function="RMSE", verbose=False, random_seed=RANDOM_STATE, thread_count=-1)
lgbm = LGBMRegressor(**study_lgbm.best_params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
extra = ExtraTreesRegressor(**study_extra.best_params, random_state=RANDOM_STATE, n_jobs=-1)
rf = RandomForestRegressor(**study_rf.best_params, random_state=RANDOM_STATE, n_jobs=-1)
ridge = make_pipeline(RobustScaler(), Ridge(alpha=study_ridge.best_params["alpha"]))

models = {"xgb": xgb, "cat": cat, "lgbm": lgbm, "extra": extra, "rf": rf, "ridge": ridge}
model_names = list(models.keys())
oof_predictions = {}

for name, model in models.items():
    preds = cross_val_predict(model, X, y_log, cv=cv, n_jobs=1, method="predict")
    oof_predictions[name] = preds
    rmse = root_mean_squared_error(y_log, preds)
    print(name, rmse)

OOF = np.column_stack([oof_predictions[name] for name in model_names])

def objective_weights(trial):
    raw_weights = np.array([trial.suggest_float(f"weight_{name}", 0.0, 1.0) for name in model_names])
    if raw_weights.sum() == 0:
        return float("inf")
    weights = raw_weights / raw_weights.sum()
    blended_pred = OOF @ weights
    return root_mean_squared_error(y_log, blended_pred)

study_weights = optuna.create_study(direction="minimize")
study_weights.optimize(objective_weights, n_trials=N_TRIALS_WEIGHTS)

raw_weights = np.array([study_weights.best_params[f"weight_{name}"] for name in model_names])
best_weights = raw_weights / raw_weights.sum()

print("Ensemble Log-RMSE:", study_weights.best_value)

for name, weight in zip(model_names, best_weights):
    print(name, weight)

for name, model in models.items():
    model.fit(X, y_log)

def predict_ensemble(X_new):
    log_predictions = np.column_stack([models[name].predict(X_new) for name in model_names])
    ensemble_log_pred = log_predictions @ best_weights
    prediction = np.expm1(ensemble_log_pred)
    return np.maximum(prediction, 0)

final_predictions = predict_ensemble(df_test_processed)

[I 2026-08-31 16:34:40,340] A new study created in memory with name: no-name-d799a5d6-b23c-4778-a991-532a3456bf97
[I 2026-08-31 16:34:42,314] Trial 0 finished with value: 0.19862432209736594 and parameters: {'n_estimators': 1241, 'learning_rate': 0.0196135765160949, 'max_depth': 7, 'min_child_weight': 16.18983225861832, 'subsample': 0.8488028500705318, 'colsample_bytree': 0.8133327206142316, 'gamma': 4.911702294828157, 'reg_alpha': 0.21367786740954467, 'reg_lambda': 32.73181461616147}. Best is trial 0 with value: 0.19862432209736594.
[I 2026-08-31 16:34:45,504] Trial 1 finished with value: 0.14639614632768444 and parameters: {'n_estimators': 2020, 'learning_rate': 0.008212423415085728, 'max_depth': 2, 'min_child_weight': 10.141824261611037, 'subsample': 0.8171685015939061, 'colsample_bytree': 0.7886839807193383, 'gamma': 0.907377020090015, 'reg_alpha': 0.034058176812655466, 'reg_lambda': 0.10560713339114987}. Best is trial 1 with value: 0.14639614632768444.
[I 2026-08-31 16:34:53,511] 

XGB: 0.12951399165196417 {'n_estimators': 3510, 'learning_rate': 0.012673129862021857, 'max_depth': 2, 'min_child_weight': 1.2505259810096563, 'subsample': 0.594597720267598, 'colsample_bytree': 0.5630076573853168, 'gamma': 0.012437317526691125, 'reg_alpha': 0.00018277278780590292, 'reg_lambda': 1.8956696743706758}
CAT: 0.1237922658364399 {'iterations': 4149, 'learning_rate': 0.01022175935989865, 'depth': 5, 'l2_leaf_reg': 0.4915847295618146, 'random_strength': 3.2639137479973677, 'bagging_temperature': 4.588129606191085}
LGBM: 0.13034198324420612 {'n_estimators': 4918, 'learning_rate': 0.005419374334001255, 'num_leaves': 28, 'max_depth': 7, 'min_child_samples': 72, 'subsample': 0.909383813683474, 'colsample_bytree': 0.5126348076943419, 'reg_alpha': 0.2950835104737151, 'reg_lambda': 10.450344504905214}
EXTRA: 0.13398946426256628 {'n_estimators': 309, 'max_depth': 30, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.5283758154997131}
RF: 0.13827329698613822 {'n_estimator

[I 2026-08-31 17:50:13,391] A new study created in memory with name: no-name-c3e771b0-8d2c-4405-84ee-16d8b16608f2
[I 2026-08-31 17:50:13,394] Trial 0 finished with value: 0.13014540480688078 and parameters: {'weight_xgb': 0.058456563592096944, 'weight_cat': 0.2113201596083889, 'weight_lgbm': 0.5320138914724676, 'weight_extra': 0.9501388338662703, 'weight_rf': 0.6789815427282804, 'weight_ridge': 0.45921429721793416}. Best is trial 0 with value: 0.13014540480688078.
[I 2026-08-31 17:50:13,396] Trial 1 finished with value: 0.12868207649304783 and parameters: {'weight_xgb': 0.6401383508214569, 'weight_cat': 0.7892542867138265, 'weight_lgbm': 0.6909574843291468, 'weight_extra': 0.758492456527342, 'weight_rf': 0.8902428223478889, 'weight_ridge': 0.762078443434941}. Best is trial 1 with value: 0.12868207649304783.
[I 2026-08-31 17:50:13,398] Trial 2 finished with value: 0.13040388612963544 and parameters: {'weight_xgb': 0.8701291021438323, 'weight_cat': 0.25561606743629783, 'weight_lgbm': 0.7

rf 0.13901984182095914
ridge 0.15767339279375733


[I 2026-08-31 17:50:13,412] Trial 9 finished with value: 0.13238011681162243 and parameters: {'weight_xgb': 0.05819783338296003, 'weight_cat': 0.05218436056047571, 'weight_lgbm': 0.40439941800617885, 'weight_extra': 0.649088251102946, 'weight_rf': 0.7900310293603711, 'weight_ridge': 0.743711174334419}. Best is trial 3 with value: 0.12680314800100817.
[I 2026-08-31 17:50:13,423] Trial 10 finished with value: 0.12695485706113527 and parameters: {'weight_xgb': 0.44340158494519977, 'weight_cat': 0.5204207733713507, 'weight_lgbm': 0.9786157714751463, 'weight_extra': 0.3611090500270983, 'weight_rf': 0.590880167217384, 'weight_ridge': 0.005444772054657529}. Best is trial 3 with value: 0.12680314800100817.
[I 2026-08-31 17:50:13,434] Trial 11 finished with value: 0.12686369789646232 and parameters: {'weight_xgb': 0.4230392106228082, 'weight_cat': 0.5123500330024002, 'weight_lgbm': 0.9815154784142482, 'weight_extra': 0.39654402765779784, 'weight_rf': 0.5236915033719984, 'weight_ridge': 0.012138

Ensemble Log-RMSE: 0.1248676112590638
xgb 0.1235547219739923
cat 0.4372169813427838
lgbm 0.2021073411843538
extra 0.23581148866197876
rf 0.00017066648139314175
ridge 0.001138800355498109


In [ ]:
from xgboost import XGBRegressor

XGB_best_model = XGBRegressor(
    **study.best_params,
    random_state=42
)

score_dataset(X, y, XGB_best_model)

In [ ]:
XGB_best_model2 = XGBRegressor(
    **study.best_params,
    random_state=43
)

score_dataset(X, y, XGB_best_model2)

In [23]:
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# --------------------------------------------------
# Chosen hyperparameters
# --------------------------------------------------

XGB_PARAMS = {
    "n_estimators": 3510,
    "learning_rate": 0.012673129862021857,
    "max_depth": 2,
    "min_child_weight": 1.2505259810096563,
    "subsample": 0.594597720267598,
    "colsample_bytree": 0.5630076573853168,
    "gamma": 0.012437317526691125,
    "reg_alpha": 0.00018277278780590292,
    "reg_lambda": 1.8956696743706758
}

CAT_PARAMS = {
    "iterations": 4149,
    "learning_rate": 0.01022175935989865,
    "depth": 5,
    "l2_leaf_reg": 0.4915847295618146,
    "random_strength": 3.2639137479973677,
    "bagging_temperature": 4.588129606191085
}

LGBM_PARAMS = {
    "n_estimators": 4918,
    "learning_rate": 0.005419374334001255,
    "num_leaves": 28,
    "max_depth": 7,
    "min_child_samples": 72,
    "subsample": 0.909383813683474,
    "colsample_bytree": 0.5126348076943419,
    "reg_alpha": 0.2950835104737151,
    "reg_lambda": 10.450344504905214
}

EXTRA_PARAMS = {
    "n_estimators": 309,
    "max_depth": 30,
    "min_samples_split": 6,
    "min_samples_leaf": 3,
    "max_features": 0.5283758154997131
}

RF_PARAMS = {
    "n_estimators": 1224,
    "max_depth": 35,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": 0.6678443223954026
}

RIDGE_ALPHA = 0.6322971673506648


# --------------------------------------------------
# Ensemble weights from your output
# --------------------------------------------------

BEST_WEIGHTS = np.array([
    0.1235547219739923,   # XGB
    0.4372169813427838,   # CAT
    0.2021073411843538,   # LGBM
    0.23581148866197876,  # EXTRA
    0.00017066648139314175, # RF
    0.001138800355498109    # RIDGE
])

# Normalize just in case
BEST_WEIGHTS = BEST_WEIGHTS / BEST_WEIGHTS.sum()


# Your target is trained in log space
y_log = np.log1p(np.asarray(y))


def train_and_predict(seed):

    print(f"\nTraining seed {seed}")

    xgb = XGBRegressor(
        **XGB_PARAMS,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    cat = CatBoostRegressor(
        **CAT_PARAMS,
        loss_function="RMSE",
        verbose=False,
        random_seed=seed,
        thread_count=-1
    )

    lgbm = LGBMRegressor(
        **LGBM_PARAMS,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

    extra = ExtraTreesRegressor(
        **EXTRA_PARAMS,
        random_state=seed,
        n_jobs=-1
    )

    rf = RandomForestRegressor(
        **RF_PARAMS,
        random_state=seed,
        n_jobs=-1
    )

    ridge = make_pipeline(
        RobustScaler(),
        Ridge(alpha=RIDGE_ALPHA)
    )

    models = [
        xgb,
        cat,
        lgbm,
        extra,
        rf,
        ridge
    ]

    log_predictions = []

    for model in models:

        model.fit(X, y_log)

        pred = model.predict(df_test_processed)

        log_predictions.append(pred)

    log_predictions = np.column_stack(log_predictions)

    # Weighted ensemble in log space
    ensemble_log_pred = log_predictions @ BEST_WEIGHTS

    # Convert back to actual SalePrice
    predictions = np.expm1(ensemble_log_pred)

    return np.maximum(predictions, 0)


# --------------------------------------------------
# Run three seeds
# --------------------------------------------------

predictions_final_1 = train_and_predict(42)

predictions_final_2 = train_and_predict(123)

predictions_final_3 = train_and_predict(999)


# --------------------------------------------------
# Average the 3 ensembles
# --------------------------------------------------

predictions_final = (
    predictions_final_1 +
    predictions_final_2 +
    predictions_final_3
) / 3


print("Done")
print(predictions_final.shape)


Training seed 42

Training seed 123

Training seed 999
Done
(1459,)


In [ ]:
print("All features:", score_dataset(X, y))

for threshold in [0, 0.005, 0.01, 0.02, 0.05]:
    features = mi_scores[mi_scores > threshold].index
    print(threshold, len(features), score_dataset(X[features], y))

In [26]:

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": predictions_final_3
})

submission.to_csv("submissionensemble5.csv", index=False)

In [27]:
import numpy as np
import optuna

from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 30
N_TRIALS_WEIGHTS = 200

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

y_log = np.log1p(np.asarray(y))

def cv_rmse(model):
    scores = cross_val_score(
        model,
        X,
        y_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )
    return -scores.mean()

def objective_xgb(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 1000, 6000),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        min_child_weight=trial.suggest_float("min_child_weight", 0.1, 10, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 0.9),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 0.9),
        gamma=trial.suggest_float("gamma", 0.0, 2.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 2, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 20, log=True),
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    return cv_rmse(model)

def objective_cat(trial):
    model = CatBoostRegressor(
        iterations=trial.suggest_int("iterations", 1000, 6000),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        depth=trial.suggest_int("depth", 4, 7),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 0.05, 20, log=True),
        random_strength=trial.suggest_float("random_strength", 1e-3, 5, log=True),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0, 10),
        loss_function="RMSE",
        verbose=False,
        random_seed=RANDOM_STATE,
        thread_count=-1
    )
    return cv_rmse(model)

def objective_lgbm(trial):
    model = LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 1000, 6000),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 50),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 100),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 0.9),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )
    return cv_rmse(model)

def objective_gbr(trial):
    model = GradientBoostingRegressor(
        n_estimators=trial.suggest_int("n_estimators", 500, 4000),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 30),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        loss=trial.suggest_categorical("loss", ["huber", "squared_error"]),
        random_state=RANDOM_STATE
    )
    return cv_rmse(model)

def objective_elastic(trial):
    model = make_pipeline(
        RobustScaler(),
        ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-5, 0.1, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.05, 1.0),
            max_iter=20000,
            random_state=RANDOM_STATE
        )
    )
    return cv_rmse(model)

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)

study_gbr = optuna.create_study(direction="minimize")
study_gbr.optimize(objective_gbr, n_trials=N_TRIALS)

study_elastic = optuna.create_study(direction="minimize")
study_elastic.optimize(objective_elastic, n_trials=N_TRIALS)

print("XGB:", study_xgb.best_value, study_xgb.best_params)
print("CAT:", study_cat.best_value, study_cat.best_params)
print("LGBM:", study_lgbm.best_value, study_lgbm.best_params)
print("GBR:", study_gbr.best_value, study_gbr.best_params)
print("ELASTIC:", study_elastic.best_value, study_elastic.best_params)

xgb = XGBRegressor(
    **study_xgb.best_params,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cat = CatBoostRegressor(
    **study_cat.best_params,
    loss_function="RMSE",
    verbose=False,
    random_seed=RANDOM_STATE,
    thread_count=-1
)

lgbm = LGBMRegressor(
    **study_lgbm.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

gbr = GradientBoostingRegressor(
    **study_gbr.best_params,
    random_state=RANDOM_STATE
)

elastic = make_pipeline(
    RobustScaler(),
    ElasticNet(
        alpha=study_elastic.best_params["alpha"],
        l1_ratio=study_elastic.best_params["l1_ratio"],
        max_iter=20000,
        random_state=RANDOM_STATE
    )
)

models = {
    "xgb": xgb,
    "cat": cat,
    "lgbm": lgbm,
    "gbr": gbr,
    "elastic": elastic
}

model_names = list(models.keys())

oof_predictions = {}

for name, model in models.items():
    preds = cross_val_predict(
        model,
        X,
        y_log,
        cv=cv,
        n_jobs=1,
        method="predict"
    )

    oof_predictions[name] = preds

    print(
        name,
        root_mean_squared_error(y_log, preds)
    )

OOF = np.column_stack([
    oof_predictions[name]
    for name in model_names
])

def objective_weights(trial):
    raw_weights = np.array([
        trial.suggest_float(
            f"weight_{name}",
            0,
            1
        )
        for name in model_names
    ])

    if raw_weights.sum() == 0:
        return float("inf")

    weights = raw_weights / raw_weights.sum()

    prediction = OOF @ weights

    return root_mean_squared_error(
        y_log,
        prediction
    )

study_weights = optuna.create_study(
    direction="minimize"
)

study_weights.optimize(
    objective_weights,
    n_trials=N_TRIALS_WEIGHTS
)

raw_weights = np.array([
    study_weights.best_params[f"weight_{name}"]
    for name in model_names
])

best_weights = raw_weights / raw_weights.sum()

print(
    "Ensemble Log-RMSE:",
    study_weights.best_value
)

for name, weight in zip(
    model_names,
    best_weights
):
    print(name, weight)

for model in models.values():
    model.fit(X, y_log)

log_predictions = np.column_stack([
    models[name].predict(df_test_processed)
    for name in model_names
])

ensemble_log_predictions = (
    log_predictions @ best_weights
)

final_predictions = np.maximum(
    np.expm1(ensemble_log_predictions),
    0
)

print(final_predictions.shape)

[I 2026-08-31 20:07:10,471] A new study created in memory with name: no-name-d0e9d939-2627-4a3a-8427-5c15f7434461
[I 2026-08-31 20:07:19,446] Trial 0 finished with value: 0.1550835250227504 and parameters: {'n_estimators': 3878, 'learning_rate': 0.004095517102628475, 'max_depth': 3, 'min_child_weight': 6.2831865500660085, 'subsample': 0.5953821009051355, 'colsample_bytree': 0.8966016129773221, 'gamma': 1.5036686869892297, 'reg_alpha': 0.000275953504326452, 'reg_lambda': 1.8328434215094016}. Best is trial 0 with value: 0.1550835250227504.
[I 2026-08-31 20:07:28,745] Trial 1 finished with value: 0.13856600650910939 and parameters: {'n_estimators': 4208, 'learning_rate': 0.003704457501890289, 'max_depth': 3, 'min_child_weight': 0.36359417492360085, 'subsample': 0.7912239117752395, 'colsample_bytree': 0.6419591140464089, 'gamma': 0.30467388355096725, 'reg_alpha': 0.3722252382413988, 'reg_lambda': 8.338867894596989}. Best is trial 1 with value: 0.13856600650910939.
[I 2026-08-31 20:07:39,50

XGB: 0.12930499277561447 {'n_estimators': 1513, 'learning_rate': 0.025062361836477583, 'max_depth': 3, 'min_child_weight': 1.6983893030290647, 'subsample': 0.8603805272092565, 'colsample_bytree': 0.744304731223331, 'gamma': 0.0001579074785360672, 'reg_alpha': 0.00011921318833886257, 'reg_lambda': 0.8029281069659692}
CAT: 0.12355888574071094 {'iterations': 4187, 'learning_rate': 0.004109054956033522, 'depth': 6, 'l2_leaf_reg': 0.35658455151507984, 'random_strength': 4.706979541014375, 'bagging_temperature': 5.950159666491976}
LGBM: 0.1294308412332504 {'n_estimators': 3492, 'learning_rate': 0.006844292325219354, 'num_leaves': 36, 'max_depth': 4, 'min_child_samples': 21, 'subsample': 0.5879917744659336, 'colsample_bytree': 0.4625177254406786, 'reg_alpha': 0.08153368237146785, 'reg_lambda': 0.02675810935391093}
GBR: 0.1256026095391342 {'n_estimators': 3938, 'learning_rate': 0.012261053653303986, 'max_depth': 3, 'min_samples_split': 19, 'min_samples_leaf': 29, 'max_features': 0.373278320378

[I 2026-08-31 21:40:40,042] A new study created in memory with name: no-name-73772f1c-8317-48e2-8df8-8b61f0965353
[I 2026-08-31 21:40:40,045] Trial 0 finished with value: 0.12703206352560456 and parameters: {'weight_xgb': 0.06830773872941642, 'weight_cat': 0.6213489696620172, 'weight_lgbm': 0.5165165890319323, 'weight_gbr': 0.5275415856084571, 'weight_elastic': 0.3281818835227336}. Best is trial 0 with value: 0.12703206352560456.
[I 2026-08-31 21:40:40,047] Trial 1 finished with value: 0.12572100380885343 and parameters: {'weight_xgb': 0.6994390878517612, 'weight_cat': 0.5947069886440715, 'weight_lgbm': 0.34577727732765096, 'weight_gbr': 0.7787148746217725, 'weight_elastic': 0.03479659887701725}. Best is trial 1 with value: 0.12572100380885343.
[I 2026-08-31 21:40:40,049] Trial 2 finished with value: 0.12719945444806777 and parameters: {'weight_xgb': 0.2818112022996111, 'weight_cat': 0.39294645988910715, 'weight_lgbm': 0.4791326683700342, 'weight_gbr': 0.24716477782221036, 'weight_elas

elastic 0.15787301374023335


[I 2026-08-31 21:40:40,248] Trial 29 finished with value: 0.12708508536848342 and parameters: {'weight_xgb': 0.6426582069326364, 'weight_cat': 0.6623334965406594, 'weight_lgbm': 0.18127580477505117, 'weight_gbr': 0.46105336298274363, 'weight_elastic': 0.3183638762474521}. Best is trial 25 with value: 0.125204830789606.
[I 2026-08-31 21:40:40,258] Trial 30 finished with value: 0.12652437907200528 and parameters: {'weight_xgb': 0.48905100573892624, 'weight_cat': 0.2857068201845344, 'weight_lgbm': 0.0019606319413348616, 'weight_gbr': 0.1704318359537082, 'weight_elastic': 0.04847359327164903}. Best is trial 25 with value: 0.125204830789606.
[I 2026-08-31 21:40:40,267] Trial 31 finished with value: 0.1255236644995144 and parameters: {'weight_xgb': 0.8023249885951594, 'weight_cat': 0.6987854470197488, 'weight_lgbm': 0.27725462501724507, 'weight_gbr': 0.9402209303203155, 'weight_elastic': 0.022006631844239982}. Best is trial 25 with value: 0.125204830789606.
[I 2026-08-31 21:40:40,275] Trial 

Ensemble Log-RMSE: 0.12368218768499746
xgb 0.024455164518109834
cat 0.7042601175156455
lgbm 0.015156268927028113
gbr 0.2548107002342416
elastic 0.0013177488049749286
(1459,)


In [28]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_predictions
})

submission.to_csv("submissionensemble6.csv", index=False)

In [29]:
X.to_csv("Xtillyet.csv",index=False)

In [ ]:
df_test_processed.to_csv("dftesttillyet.csv",index = False)
X.to_csv("Xtillyet.csv",index=False)

In [5]:
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import root_mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


def generate_ensemble_predictions(X, y, df_test, random_state=42, n_splits=5):

    y_log = np.log1p(np.asarray(y))

    xgb = XGBRegressor(
        n_estimators=1513,
        learning_rate=0.025062361836477583,
        max_depth=3,
        min_child_weight=1.6983893030290647,
        subsample=0.8603805272092565,
        colsample_bytree=0.744304731223331,
        gamma=0.0001579074785360672,
        reg_alpha=0.00011921318833886257,
        reg_lambda=0.8029281069659692,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1
    )

    cat = CatBoostRegressor(
        iterations=4187,
        learning_rate=0.004109054956033522,
        depth=6,
        l2_leaf_reg=0.35658455151507984,
        random_strength=4.706979541014375,
        bagging_temperature=5.950159666491976,
        loss_function="RMSE",
        verbose=False,
        random_seed=random_state,
        thread_count=-1
    )

    lgbm = LGBMRegressor(
        n_estimators=3492,
        learning_rate=0.006844292325219354,
        num_leaves=36,
        max_depth=4,
        min_child_samples=21,
        subsample=0.5879917744659336,
        colsample_bytree=0.4625177254406786,
        reg_alpha=0.08153368237146785,
        reg_lambda=0.02675810935391093,
        random_state=random_state,
        n_jobs=-1,
        verbosity=-1
    )

    gbr = GradientBoostingRegressor(
        n_estimators=3938,
        learning_rate=0.012261053653303986,
        max_depth=3,
        min_samples_split=19,
        min_samples_leaf=29,
        max_features=0.37327832037835607,
        loss="huber",
        random_state=random_state
    )

    models = {
        "xgb": xgb,
        "cat": cat,
        "lgbm": lgbm,
        "gbr": gbr
    }

    weights = {
        "xgb": 0.024455164518109834,
        "cat": 0.7042601175156455,
        "lgbm": 0.015156268927028113,
        "gbr": 0.2548107002342416
    }

    weight_array = np.array([
        weights["xgb"],
        weights["cat"],
        weights["lgbm"],
        weights["gbr"]
    ])

    weight_array = weight_array / weight_array.sum()

    cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    oof_predictions = []

    for name, model in models.items():
        preds = cross_val_predict(
            model,
            X,
            y_log,
            cv=cv,
            n_jobs=1
        )

        oof_predictions.append(preds)

        score = root_mean_squared_error(
            y_log,
            preds
        )

        print(name, score)

    oof_predictions = np.column_stack(oof_predictions)

    ensemble_oof = oof_predictions @ weight_array

    ensemble_score = root_mean_squared_error(
        y_log,
        ensemble_oof
    )

    print("Ensemble Log-RMSE:", ensemble_score)

    test_predictions = []

    for name, model in models.items():
        model.fit(X, y_log)
        pred = model.predict(df_test)
        test_predictions.append(pred)

    test_predictions = np.column_stack(test_predictions)

    ensemble_log_pred = test_predictions @ weight_array

    predictions = np.expm1(ensemble_log_pred)

    return np.maximum(predictions, 0), ensemble_score

In [41]:
predictions,score = generate_ensemble_predictions(X, y, df_test_processed)

xgb 0.13074340773381776
cat 0.12433865768993975
lgbm 0.1304155994620408
gbr 0.12671623519644737
elastic 0.15787301374023335
Ensemble Log-RMSE: 0.12368218768499746


In [42]:
score

0.12368218768499746

In [4]:
df_test_processed= pd.read_csv("dftesttillyet.csv")
X = pd.read_csv("Xtillyet.csv")

In [6]:
drop_features = [
    "QualNeighborhoodArea",
    "Ageatsale",
    "Garageageatsale",
    "AreaPerRoom"
]

X = X.drop(
    columns=[col for col in drop_features if col in X.columns]
)

df_test_processed = df_test_processed.drop(
    columns=[col for col in drop_features if col in df_test_processed.columns]
)

In [8]:
for df in [X, df_test_processed]:

    df["TotalSF"] = (
        df.get("TotalBsmtSF", 0)
        + df.get("1stFlrSF", 0)
        + df.get("2ndFlrSF", 0)
    )

    df["TotalBathrooms"] = (
        df.get("FullBath", 0)
        + 0.5 * df.get("HalfBath", 0)
        + df.get("BsmtFullBath", 0)
        + 0.5 * df.get("BsmtHalfBath", 0)
    )

    df["TotalPorchSF"] = (
        df.get("WoodDeckSF", 0)
        + df.get("OpenPorchSF", 0)
        + df.get("EnclosedPorch", 0)
        + df.get("3SsnPorch", 0)
        + df.get("ScreenPorch", 0)
    )

    df["HouseAge"] = (
        df.get("YrSold", 0)
        - df.get("YearBuilt", 0)
    )

    df["RemodAge"] = (
        df.get("YrSold", 0)
        - df.get("YearRemodAdd", 0)
    )

    df["GarageAge"] = (
        df.get("YrSold", 0)
        - df.get("GarageYrBlt", 0)
    )

    df["GarageScore"] = (
        df.get("GarageCars", 0)
        * df.get("GarageArea", 0)
    )

    df["OverallQual_GrLivArea"] = (
        df.get("OverallQual", 0)
        * df.get("GrLivArea", 0)
    )

    df["OverallQual_TotalSF"] = (
        df.get("OverallQual", 0)
        * df["TotalSF"]
    )

    df["OverallQualSquared"] = (
        df.get("OverallQual", 0) ** 2
    )

    df["HasGarage"] = (
        df.get("GarageArea", 0) > 0
    ).astype(int)

    df["HasBsmt"] = (
        df.get("TotalBsmtSF", 0) > 0
    ).astype(int)

In [10]:
df_test_processed.to_csv("dftesttillyetmore.csv",index = False)
X.to_csv("Xtillyetmore.csv",index=False)

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

size_features = [
    "GrLivArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "GarageArea",
    "LotArea",
    "TotalSF",
    "TotalPorchSF"
]

scaler_size = StandardScaler()

X_size_scaled = scaler_size.fit_transform(X[size_features])
test_size_scaled = scaler_size.transform(df_test_processed[size_features])

pca_size = PCA(n_components=3, random_state=42)

X_size_pca = pca_size.fit_transform(X_size_scaled)
test_size_pca = pca_size.transform(test_size_scaled)

for i in range(3):
    X[f"SizePCA_{i+1}"] = X_size_pca[:, i]
    df_test_processed[f"SizePCA_{i+1}"] = test_size_pca[:, i]

In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def add_pca_features(X_train, X_test, columns, prefix, n_components=2):

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(X_train[columns])
    test_scaled = scaler.transform(X_test[columns])

    pca = PCA(
        n_components=n_components,
        random_state=42
    )

    train_pca = pca.fit_transform(train_scaled)
    test_pca = pca.transform(test_scaled)

    for i in range(n_components):
        X_train[f"{prefix}_PCA{i+1}"] = train_pca[:, i]
        X_test[f"{prefix}_PCA{i+1}"] = test_pca[:, i]

    print(
        prefix,
        "explained variance:",
        pca.explained_variance_ratio_
    )

    

In [13]:

quality_features = [
    "OverallQual",
    "ExterQual",
    "KitchenQual",
    "BsmtQual",
    "GarageFinish",
    "HeatingQC"
]

age_features = [
    "YearBuilt",
    "GarageYrBlt",
    "HouseAge",
    "RemodAge",
]

In [14]:
add_pca_features(
    X,
    df_test_processed,
    quality_features,
    "Quality",
    2
)

add_pca_features(
    X,
    df_test_processed,
    age_features,
    "Age",
    2
)

Quality explained variance: [0.57944882 0.13068319]
Age explained variance: [0.93185974 0.06814026]


In [15]:
cluster_features = [
    "OverallQual",
    "GrLivArea",
    "TotalSF",
    "YearBuilt",
    "GarageArea",
    "TotalBathrooms",
    "LotArea",
    "TotRmsAbvGrd"
]

In [16]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_features = [
    "OverallQual",
    "GrLivArea",
    "TotalSF",
    "YearBuilt",
    "GarageArea",
    "TotalBathrooms",
    "LotArea",
    "TotRmsAbvGrd"
]

cluster_scaler = StandardScaler()

X_cluster_scaled = cluster_scaler.fit_transform(
    X[cluster_features]
)

test_cluster_scaled = cluster_scaler.transform(
    df_test_processed[cluster_features]
)

kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

X["HouseCluster"] = kmeans.fit_predict(
    X_cluster_scaled
)

df_test_processed["HouseCluster"] = kmeans.predict(
    test_cluster_scaled
)

In [17]:
train_distances = kmeans.transform(X_cluster_scaled)
test_distances = kmeans.transform(test_cluster_scaled)

for i in range(kmeans.n_clusters):

    X[f"ClusterDist_{i}"] = train_distances[:, i]

    df_test_processed[f"ClusterDist_{i}"] = (
        test_distances[:, i]
    )

In [18]:
features = [
    "OverallQual",
    "GrLivArea",
    "TotalSF",
    "YearBuilt",
    "GarageArea",
    "TotalBathrooms",
    "LotArea",
    "TotRmsAbvGrd",
    "TotalBsmtSF",
    "OverallCond"
]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X[features])
test_scaled = scaler.transform(df_test_processed[features])

pca = PCA(n_components=5, random_state=42)

X_pc = pca.fit_transform(X_scaled)
test_pc = pca.transform(test_scaled)

kmeans = KMeans(
    n_clusters=6,
    n_init=20,
    random_state=42
)

X["PCACluster"] = kmeans.fit_predict(X_pc)
df_test_processed["PCACluster"] = kmeans.predict(test_pc)

train_dist = kmeans.transform(X_pc)
test_dist = kmeans.transform(test_pc)

for i in range(6):
    X[f"PCAClusterDist_{i}"] = train_dist[:, i]
    df_test_processed[f"PCAClusterDist_{i}"] = test_dist[:, i]

In [19]:
import numpy as np
import optuna

from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 30
N_TRIALS_WEIGHTS = 200

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

y_log = np.log1p(np.asarray(y))

def cv_rmse(model):
    scores = cross_val_score(
        model,
        X,
        y_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1
    )
    return -scores.mean()

def objective_xgb(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 1000, 6000),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        min_child_weight=trial.suggest_float("min_child_weight", 0.1, 10, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 0.9),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 0.9),
        gamma=trial.suggest_float("gamma", 0.0, 2.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 2, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 20, log=True),
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    return cv_rmse(model)

def objective_cat(trial):
    model = CatBoostRegressor(
        iterations=trial.suggest_int("iterations", 1000, 6000),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        depth=trial.suggest_int("depth", 4, 7),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 0.05, 20, log=True),
        random_strength=trial.suggest_float("random_strength", 1e-3, 5, log=True),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0, 10),
        loss_function="RMSE",
        verbose=False,
        random_seed=RANDOM_STATE,
        thread_count=-1
    )
    return cv_rmse(model)

def objective_lgbm(trial):
    model = LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 1000, 6000),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 50),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 100),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 0.9),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )
    return cv_rmse(model)

def objective_gbr(trial):
    model = GradientBoostingRegressor(
        n_estimators=trial.suggest_int("n_estimators", 500, 4000),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 30),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        loss=trial.suggest_categorical("loss", ["huber", "squared_error"]),
        random_state=RANDOM_STATE
    )
    return cv_rmse(model)

def objective_elastic(trial):
    model = make_pipeline(
        RobustScaler(),
        ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-5, 0.1, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.05, 1.0),
            max_iter=20000,
            random_state=RANDOM_STATE
        )
    )
    return cv_rmse(model)

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

study_cat = optuna.create_study(direction="minimize")
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)

study_gbr = optuna.create_study(direction="minimize")
study_gbr.optimize(objective_gbr, n_trials=N_TRIALS)

study_elastic = optuna.create_study(direction="minimize")
study_elastic.optimize(objective_elastic, n_trials=N_TRIALS)

print("XGB:", study_xgb.best_value, study_xgb.best_params)
print("CAT:", study_cat.best_value, study_cat.best_params)
print("LGBM:", study_lgbm.best_value, study_lgbm.best_params)
print("GBR:", study_gbr.best_value, study_gbr.best_params)
print("ELASTIC:", study_elastic.best_value, study_elastic.best_params)

xgb = XGBRegressor(
    **study_xgb.best_params,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cat = CatBoostRegressor(
    **study_cat.best_params,
    loss_function="RMSE",
    verbose=False,
    random_seed=RANDOM_STATE,
    thread_count=-1
)

lgbm = LGBMRegressor(
    **study_lgbm.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

gbr = GradientBoostingRegressor(
    **study_gbr.best_params,
    random_state=RANDOM_STATE
)

elastic = make_pipeline(
    RobustScaler(),
    ElasticNet(
        alpha=study_elastic.best_params["alpha"],
        l1_ratio=study_elastic.best_params["l1_ratio"],
        max_iter=20000,
        random_state=RANDOM_STATE
    )
)

models = {
    "xgb": xgb,
    "cat": cat,
    "lgbm": lgbm,
    "gbr": gbr,
    "elastic": elastic
}

model_names = list(models.keys())

oof_predictions = {}

for name, model in models.items():
    preds = cross_val_predict(
        model,
        X,
        y_log,
        cv=cv,
        n_jobs=1,
        method="predict"
    )

    oof_predictions[name] = preds

    print(
        name,
        root_mean_squared_error(y_log, preds)
    )

OOF = np.column_stack([
    oof_predictions[name]
    for name in model_names
])

def objective_weights(trial):
    raw_weights = np.array([
        trial.suggest_float(
            f"weight_{name}",
            0,
            1
        )
        for name in model_names
    ])

    if raw_weights.sum() == 0:
        return float("inf")

    weights = raw_weights / raw_weights.sum()

    prediction = OOF @ weights

    return root_mean_squared_error(
        y_log,
        prediction
    )

study_weights = optuna.create_study(
    direction="minimize"
)

study_weights.optimize(
    objective_weights,
    n_trials=N_TRIALS_WEIGHTS
)

raw_weights = np.array([
    study_weights.best_params[f"weight_{name}"]
    for name in model_names
])

best_weights = raw_weights / raw_weights.sum()

print(
    "Ensemble Log-RMSE:",
    study_weights.best_value
)

for name, weight in zip(
    model_names,
    best_weights
):
    print(name, weight)

for model in models.values():
    model.fit(X, y_log)

log_predictions = np.column_stack([
    models[name].predict(df_test_processed)
    for name in model_names
])

ensemble_log_predictions = (
    log_predictions @ best_weights
)

final_predictions = np.maximum(
    np.expm1(ensemble_log_predictions),
    0
)

print(final_predictions.shape)

[I 2026-08-31 23:07:52,158] A new study created in memory with name: no-name-b8a87bd1-52bc-4f05-932c-f43d1de85bd4
[I 2026-08-31 23:07:56,725] Trial 0 finished with value: 0.15577463442842646 and parameters: {'n_estimators': 2212, 'learning_rate': 0.004570657075399112, 'max_depth': 2, 'min_child_weight': 2.6877731765899573, 'subsample': 0.8944643486507762, 'colsample_bytree': 0.4687086212268676, 'gamma': 1.76867311096868, 'reg_alpha': 0.20979184503797318, 'reg_lambda': 0.2786038244742549}. Best is trial 0 with value: 0.15577463442842646.
[I 2026-08-31 23:08:07,969] Trial 1 finished with value: 0.14823075844789985 and parameters: {'n_estimators': 5928, 'learning_rate': 0.03298049144211431, 'max_depth': 4, 'min_child_weight': 0.176031692640067, 'subsample': 0.8997829664433559, 'colsample_bytree': 0.6246730207146604, 'gamma': 1.1109082693899779, 'reg_alpha': 0.06088869043365208, 'reg_lambda': 10.726057452483673}. Best is trial 1 with value: 0.14823075844789985.
[I 2026-08-31 23:08:28,693] 

XGB: 0.127686967138875 {'n_estimators': 5698, 'learning_rate': 0.006686215486507799, 'max_depth': 4, 'min_child_weight': 0.6570851025002505, 'subsample': 0.6136032171285548, 'colsample_bytree': 0.5304379997732709, 'gamma': 0.027235345437090108, 'reg_alpha': 0.006248845009884299, 'reg_lambda': 5.021957657239438}
CAT: 0.12402067424755829 {'iterations': 3494, 'learning_rate': 0.005709730045996076, 'depth': 7, 'l2_leaf_reg': 1.8515889610761422, 'random_strength': 2.2094728233320593, 'bagging_temperature': 4.982018990474846}
LGBM: 0.13031678307386113 {'n_estimators': 2056, 'learning_rate': 0.011651978635932467, 'num_leaves': 43, 'max_depth': 4, 'min_child_samples': 20, 'subsample': 0.7701512511689997, 'colsample_bytree': 0.7611418996390179, 'reg_alpha': 1.0917066991289641, 'reg_lambda': 0.005021640860174141}
GBR: 0.12663605773526582 {'n_estimators': 995, 'learning_rate': 0.014452102202061067, 'max_depth': 3, 'min_samples_split': 19, 'min_samples_leaf': 25, 'max_features': 0.369368724550519,

[I 2026-09-01 00:27:53,900] A new study created in memory with name: no-name-c386f4a5-dc18-4096-9bcc-0078857df8d1
[I 2026-09-01 00:27:53,901] Trial 0 finished with value: 0.13019401657587018 and parameters: {'weight_xgb': 0.7083423956965792, 'weight_cat': 0.042901175703371375, 'weight_lgbm': 0.6263690724989542, 'weight_gbr': 0.3865177870287849, 'weight_elastic': 0.5256002042694062}. Best is trial 0 with value: 0.13019401657587018.
[I 2026-09-01 00:27:53,903] Trial 1 finished with value: 0.12869691433579633 and parameters: {'weight_xgb': 0.4790261352559775, 'weight_cat': 0.6267970052665502, 'weight_lgbm': 0.5549483045844706, 'weight_gbr': 0.4497686366634275, 'weight_elastic': 0.5480053902903795}. Best is trial 1 with value: 0.12869691433579633.
[I 2026-09-01 00:27:53,904] Trial 2 finished with value: 0.1350196828315895 and parameters: {'weight_xgb': 0.30830039602724757, 'weight_cat': 0.0817218622834629, 'weight_lgbm': 0.1637070812488104, 'weight_gbr': 0.43764705280827343, 'weight_elasti

gbr 0.12740508794812824
elastic 0.15638773670170575


[I 2026-09-01 00:27:53,988] Trial 26 finished with value: 0.12747825183381553 and parameters: {'weight_xgb': 0.5067103835977924, 'weight_cat': 0.9255861622314164, 'weight_lgbm': 0.40054371318366166, 'weight_gbr': 0.6015063075774224, 'weight_elastic': 0.452231182137304}. Best is trial 25 with value: 0.1258310316605026.
[I 2026-09-01 00:27:53,992] Trial 27 finished with value: 0.12583929000866773 and parameters: {'weight_xgb': 0.4377030039144347, 'weight_cat': 0.8466644968559542, 'weight_lgbm': 0.3367389552568013, 'weight_gbr': 0.7731260091023133, 'weight_elastic': 0.10064865555821101}. Best is trial 25 with value: 0.1258310316605026.
[I 2026-09-01 00:27:53,996] Trial 28 finished with value: 0.12656142229606787 and parameters: {'weight_xgb': 0.46457391904045986, 'weight_cat': 0.8676160544742819, 'weight_lgbm': 0.4164906894482142, 'weight_gbr': 0.7680100970109865, 'weight_elastic': 0.2584034841462539}. Best is trial 25 with value: 0.1258310316605026.
[I 2026-09-01 00:27:54,001] Trial 29 f

Ensemble Log-RMSE: 0.1245036660886969
xgb 0.005277357734481508
cat 0.5892743986946032
lgbm 0.010051877695542277
gbr 0.39535399642821845
elastic 4.2369447154647145e-05
(1459,)


In [20]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_predictions
})

submission.to_csv("submissionensemble11.csv", index=False)

In [22]:
y.to_csv("yfinal.csv",index=False)